# Convergence over time (PBMetaD bias reweighting)

This notebook replaces the earlier ad-hoc version of Step 4 in `README.md`.
It adopts the reweighting pipeline Bradley Harris used in
`ReweightedPopConv.ipynb` (8-replica ensembles) and `ReweightedPopConv_single.ipynb`
(single long trajectories), which fixes several issues with the previous
approach:

- **Restart-segment stitching.** PLUMED COLVAR/output files that were
  restarted are stitched into one monotonic trajectory (newer segments
  supersede the overlapping tail of older ones) instead of assuming the raw
  file is already a single continuous run.
- **Log-space reweighting.** `exp(beta * pb.bias)` is summed via
  `scipy.special.logsumexp` instead of naive `exp(...).sum()`, which
  overflows/underflows for large bias values.
- **Physical kT.** `beta = 1 / (k_B * T)` with `k_B` in kJ/mol/K and the
  actual simulation temperature, instead of a hardcoded `kT = 2.5`.
- **Cis/trans cutoff in radians**, `|wrapped omega| < pi/2`, with no dead
  zone between the two states (the old `+/-45`/`+/-135` degree thresholds
  left a gap that silently dropped frames from the state assignment).
- **Weight effective sample size (Kish ESS)** as an explicit diagnostic of
  how much a block/global estimate is dominated by a few frames.
- **Two population estimators, computed together**: the pb.bias-reweighted
  population (approximate, since only `pb.bias` and not the full `rbias`/
  `rct` was saved) and the raw biased-occupancy population (exact, but
  reflects the biased ensemble rather than the unbiased one).
- **Block-local + cumulative block-averaged estimators**, saved as CSVs, in
  place of the previous pickled per-walker convergence dictionaries.

**IMPORTANT (unchanged from the original caveat):** exact metadynamics
reweighting is unavailable because only `pb.bias` was saved, not `rbias`/
`rct`. The global raw-bias-reweighted result and the block-local/cumulative
results below should be interpreted as approximate diagnostics, not exact
free energies.

Part 1 processes the four single long trajectories (`omega`, `ophi`,
`ophps`, `oppc`) and writes CSVs to `single_trajectory_state_analysis/`.
Part 2 processes the two 8-replica ensembles (`omega`, `oppc`) and writes
CSVs to `omega_state_analysis/state_analysis_results/` and
`oppc_state_analysis/state_analysis_results/`. These are exactly the
folders and filenames already read by
[single_trajectory_state_analysis.ipynb](single_trajectory_state_analysis.ipynb).

Raw PLUMED output is not stored in this repo (see `README.md`); this
notebook expects `/Volumes/LaCie/STEPs/ConvergenceTest/...` to be mounted,
matching the paths used by the previous version of this notebook.

In [ ]:
from pathlib import Path
import itertools
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import logsumexp


# =====================================================================
# SHARED SETTINGS (Parts 1 and 2)
# =====================================================================

TEMPERATURE = 300.0

# PLUMED normally uses kJ/mol unless UNITS was changed.
K_B = 0.008314462618  # kJ mol^-1 K^-1
# If pb.bias is in kcal/mol instead, use K_B = 0.001987204258.

ANALYSIS_STRIDE = 20  # One fixed stride for every trajectory below.
N_BLOCKS = 40
N_FINAL_BLOCKS = 10  # Final blocks used for the late-time summary.

TIME_COLUMN = "time"
BIAS_COLUMN = "pb.bias"
OMEGA_COLUMNS = ["omega1", "omega2", "omega3"]

CIS_TRANS_CUTOFF = np.pi / 2

ALL_STATES = [
    "".join(x) for x in itertools.product(["C", "T"], repeat=3)
]

BETA = 1.0 / (K_B * TEMPERATURE)

STATE_COLORS = dict(
    zip(ALL_STATES, plt.cm.tab10.colors[: len(ALL_STATES)])
)

In [ ]:
# =====================================================================
# SHARED FUNCTIONS (Parts 1 and 2)
# =====================================================================

def read_plumed_colvar(filename, stride=1):
    """
    Read a restart-appended PLUMED COLVAR/output file.

    Each backward or repeated time step marks a restart segment. When a new
    segment overlaps an older segment, the newer segment supersedes the
    overlapping tail. The analysis stride is applied only after restart
    stitching, so it always samples the reconstructed chronological
    trajectory rather than the raw (possibly overlapping) file order.
    """
    filename = Path(filename)
    columns = None

    with open(filename, "r") as handle:
        for line in handle:
            if line.startswith("#! FIELDS"):
                columns = line.strip().split()[2:]
                break

    if columns is None:
        raise ValueError(f"No '#! FIELDS' line was found in {filename}")

    required_columns = [TIME_COLUMN, *OMEGA_COLUMNS, BIAS_COLUMN]

    missing = [c for c in required_columns if c not in columns]
    if missing:
        raise ValueError(
            f"{filename} is missing columns: {missing}\n"
            f"Available columns: {columns}"
        )

    raw = pd.read_csv(
        filename,
        sep=r"\s+",
        comment="#",
        header=None,
        names=columns,
        usecols=required_columns,
        on_bad_lines="warn",
    )

    for column in required_columns:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")

    raw = raw.dropna(subset=required_columns).reset_index(drop=True)

    if len(raw) == 0:
        raise ValueError(f"No valid rows were read from {filename}")

    # A non-increasing time step starts a new restart segment.
    restart_boundary = raw[TIME_COLUMN].diff().le(0).fillna(False)
    raw["_restart_segment"] = restart_boundary.cumsum()

    n_segments = raw["_restart_segment"].nunique()
    print(f"{filename.name}: read {len(raw):,} numerical rows in {n_segments} restart segment(s)")

    stitched = None

    for segment_number, segment in raw.groupby("_restart_segment", sort=True):
        segment = (
            segment.drop(columns="_restart_segment")
            .drop_duplicates(subset=TIME_COLUMN, keep="last")
            .reset_index(drop=True)
        )

        if not segment[TIME_COLUMN].is_monotonic_increasing:
            raise ValueError(
                f"Restart segment {segment_number} in {filename} is internally non-monotonic."
            )

        if stitched is None:
            stitched = segment.copy()
            continue

        new_start_time = segment[TIME_COLUMN].iloc[0]

        # The newly appended restart branch supersedes all older records
        # at and after its starting time.
        stitched = stitched[stitched[TIME_COLUMN] < new_start_time].copy()
        stitched = pd.concat([stitched, segment], ignore_index=True)

    stitched = (
        stitched.reset_index(drop=True)
        .drop_duplicates(subset=TIME_COLUMN, keep="last")
        .reset_index(drop=True)
    )

    if not stitched[TIME_COLUMN].is_monotonic_increasing:
        raise ValueError(f"Restart stitching did not produce monotonic time in {filename}.")

    removed_total = len(raw) - len(stitched)
    print(
        f"Rows after restart stitching: {len(stitched):,} "
        f"(removed {removed_total:,} overlapping/duplicate rows)"
    )

    # Apply the analysis stride only after reconstructing the chronological
    # trajectory.
    data = stitched.iloc[::stride].copy().reset_index(drop=True)
    print(f"Rows after analysis stride {stride}: {len(data):,}")

    return data


def assign_states(data):
    """
    Assign each frame to CCC through TTT.

    Cis:   |wrapped omega| < pi/2
    Trans: |wrapped omega| >= pi/2
    """
    omega = data[OMEGA_COLUMNS].to_numpy(dtype=float)
    omega = np.arctan2(np.sin(omega), np.cos(omega))

    letters = np.where(np.abs(omega) < CIS_TRANS_CUTOFF, "C", "T")

    result = data.copy()
    result["state"] = np.array(["".join(row) for row in letters], dtype="<U3")
    return result


def populations_to_free_energies(populations):
    """Convert a state-population dictionary to free energies and relative free energies."""
    free_energies = {}

    for state in ALL_STATES:
        probability = populations[state]
        free_energies[state] = (
            -K_B * TEMPERATURE * np.log(probability) if probability > 0 else np.inf
        )

    finite_values = [v for v in free_energies.values() if np.isfinite(v)]
    minimum = min(finite_values)

    delta_free_energies = {
        state: free_energies[state] - minimum for state in ALL_STATES
    }

    return free_energies, delta_free_energies


def analyze_interval(data, start, stop, label):
    """
    Analyze data.iloc[start:stop] using approximate raw-bias weights:

        w(t) = exp(beta * pb.bias(t))

    Calculations are performed in log space. Returns one record per state,
    carrying both the pb.bias-reweighted population/free energy (approximate)
    and the exact occupancy of the biased trajectory itself.
    """
    segment = data.iloc[start:stop]

    if len(segment) == 0:
        raise ValueError(f"Empty interval: {label}")

    states = segment["state"].to_numpy()
    bias = segment[BIAS_COLUMN].to_numpy(dtype=float)

    log_weights = BETA * bias
    log_total_weight = logsumexp(log_weights)

    populations = {}
    biased_populations = {}

    for state in ALL_STATES:
        mask = states == state
        biased_populations[state] = np.mean(mask)

        if np.any(mask):
            populations[state] = np.exp(logsumexp(log_weights[mask]) - log_total_weight)
        else:
            populations[state] = 0.0

    free_energies, delta_free_energies = populations_to_free_energies(populations)
    biased_free_energies, biased_delta_free_energies = populations_to_free_energies(
        biased_populations
    )

    # Weight-only Kish effective sample size.
    log_sum_weight_squared = logsumexp(2.0 * log_weights)
    weight_ess = np.exp(2.0 * log_total_weight - log_sum_weight_squared)

    records = []

    for state in ALL_STATES:
        records.append(
            {
                "interval": label,
                "state": state,
                "start_index": start,
                "stop_index": stop,
                "start_time": segment[TIME_COLUMN].iloc[0],
                "end_time": segment[TIME_COLUMN].iloc[-1],
                "midpoint_time": 0.5 * (
                    segment[TIME_COLUMN].iloc[0] + segment[TIME_COLUMN].iloc[-1]
                ),
                "n_frames": len(segment),
                # Approximate raw-bias-reweighted results.
                "population": populations[state],
                "free_energy": free_energies[state],
                "delta_free_energy": delta_free_energies[state],
                # Exact occupancy of the biased trajectory.
                "biased_population": biased_populations[state],
                "biased_free_energy": biased_free_energies[state],
                "biased_delta_free_energy": biased_delta_free_energies[state],
                # Weight diagnostics.
                "weight_ess": weight_ess,
                "weight_ess_fraction": weight_ess / len(segment),
                "mean_bias": np.mean(bias),
                "bias_std": np.std(bias),
                "bias_change": bias[-1] - bias[0],
            }
        )

    return records


def add_free_energies_to_table(table, population_column="population"):
    """Normalize each block's population vector (grouped by 'block') and recompute free energies."""
    result = table.copy()

    result[population_column] = result[population_column] / result.groupby("block")[
        population_column
    ].transform("sum")

    free_energy_column = "free_energy" if population_column == "population" else "biased_free_energy"
    delta_column = "delta_free_energy" if population_column == "population" else "biased_delta_free_energy"

    result[free_energy_column] = np.inf
    positive = result[population_column] > 0
    result.loc[positive, free_energy_column] = (
        -K_B * TEMPERATURE * np.log(result.loc[positive, population_column])
    )

    result[delta_column] = (
        result[free_energy_column] - result.groupby("block")[free_energy_column].transform("min")
    )

    return result


def add_free_energies_simple(population_table):
    """Normalize a single-row-per-state population table and (re)compute free energies from it."""
    result = population_table.copy()

    result["population"] = result["population"] / result["population"].sum()

    result["free_energy"] = np.where(
        result["population"] > 0,
        -K_B * TEMPERATURE * np.log(result["population"]),
        np.inf,
    )

    finite = np.isfinite(result["free_energy"])
    result["delta_free_energy"] = result["free_energy"] - result.loc[finite, "free_energy"].min()

    if "biased_population" in result.columns:
        result["biased_population"] = result["biased_population"] / result["biased_population"].sum()

        result["biased_free_energy"] = np.where(
            result["biased_population"] > 0,
            -K_B * TEMPERATURE * np.log(result["biased_population"]),
            np.inf,
        )

        finite_biased = np.isfinite(result["biased_free_energy"])
        result["biased_delta_free_energy"] = (
            result["biased_free_energy"] - result.loc[finite_biased, "biased_free_energy"].min()
        )

    return result

## Part 1: single long trajectories (omega, ophi, ophps, oppc)

No averaging is performed between these four independent runs; each one
biases a different set of collective variables (see "Naming of variants" in
`README.md`). Outputs are written to `single_trajectory_state_analysis/<system>/`.

In [ ]:
# =====================================================================
# PART 1 SETTINGS
# =====================================================================

SINGLE_BASE_DIRECTORY = Path("/Volumes/LaCie/STEPs/ConvergenceTest")

SINGLE_SYSTEM_NAMES = ["omega", "ophi", "ophps", "oppc"]

SINGLE_RESULTS_DIRECTORY = Path("single_trajectory_state_analysis")
SINGLE_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

SINGLE_INPUT_FILES = {
    system_name: SINGLE_BASE_DIRECTORY / system_name / "data" / "COLVAR"
    for system_name in SINGLE_SYSTEM_NAMES
}

print("Input files:")
missing_files = []

for system_name, input_file in SINGLE_INPUT_FILES.items():
    status = "FOUND" if input_file.exists() else "MISSING"
    print(f"{system_name:6s}: {input_file} [{status}]")
    if not input_file.exists():
        missing_files.append(input_file)

if missing_files:
    raise FileNotFoundError(
        "The following files were not found (mount the LaCie drive first):\n"
        + "\n".join(str(p) for p in missing_files)
    )

In [ ]:
def create_single_trajectory_plots(
    system_name, global_results, block_results, cumulative_results, block_minima,
    cumulative_minima, output_directory,
):
    """Global + time-dependent diagnostic plots for one single-trajectory system."""
    global_plot = global_results.set_index("state").reindex(ALL_STATES)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].bar(
        ALL_STATES, global_plot["delta_free_energy"],
        color=[STATE_COLORS[s] for s in ALL_STATES],
    )
    axes[0].set_xlabel("Cis/trans macrostate")
    axes[0].set_ylabel(r"Approximate global $\Delta F$ (kJ/mol)")
    axes[0].set_title(f"{system_name}: raw-bias-reweighted")
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].bar(
        ALL_STATES, global_plot["biased_delta_free_energy"],
        color=[STATE_COLORS[s] for s in ALL_STATES],
    )
    axes[1].set_xlabel("Cis/trans macrostate")
    axes[1].set_ylabel(r"Biased-occupancy $\Delta F$ (kJ/mol)")
    axes[1].set_title(f"{system_name}: observed trajectory")
    axes[1].grid(axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(output_directory / f"{system_name}_global_free_energies.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=True)

    first_late_block = max(0, N_BLOCKS - N_FINAL_BLOCKS)
    late_start_time = block_results.loc[
        block_results["block"] == first_late_block, "start_time"
    ].min()
    final_end_time = block_results["end_time"].max()

    for state in ALL_STATES:
        subset = block_results[block_results["state"] == state].sort_values("block")
        axes[0].plot(
            subset["midpoint_time"], subset["population"], marker="o", markersize=3,
            linewidth=1.1, label=state, color=STATE_COLORS[state],
        )

    axes[0].axvspan(late_start_time, final_end_time, color="gray", alpha=0.12, label="Final blocks")
    axes[0].set_ylabel("Block-local population")
    axes[0].set_ylim(0, 1)
    axes[0].set_title(f"{system_name}: independent block-local estimates")
    axes[0].grid(alpha=0.25)

    for state in ALL_STATES:
        subset = cumulative_results[cumulative_results["state"] == state].sort_values("block")
        axes[1].plot(
            subset["end_time"], subset["population"], marker="o", markersize=3,
            linewidth=1.8, label=state, color=STATE_COLORS[state],
        )

    axes[1].set_ylabel("Cumulative block-averaged population")
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"{system_name}: approximate population convergence")
    axes[1].grid(alpha=0.25)

    for state in ALL_STATES:
        subset = cumulative_results[cumulative_results["state"] == state].sort_values("block")
        values = subset["delta_free_energy"].replace([np.inf, -np.inf], np.nan)
        axes[2].plot(
            subset["end_time"], values, marker="o", markersize=3,
            linewidth=1.8, label=state, color=STATE_COLORS[state],
        )

    axes[2].set_xlabel("Simulation time")
    axes[2].set_ylabel(r"Cumulative $\Delta F$ (kJ/mol)")
    axes[2].set_title(f"{system_name}: cumulative macrostate free energies")
    axes[2].grid(alpha=0.25)

    axes[0].legend(title="State", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(output_directory / f"{system_name}_population_convergence.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    state_to_number = {state: number for number, state in enumerate(ALL_STATES)}

    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    axes[0].plot(block_minima["midpoint_time"], block_minima["minimum_state"].map(state_to_number), marker="o")
    axes[0].set_yticks(list(state_to_number.values()))
    axes[0].set_yticklabels(list(state_to_number.keys()))
    axes[0].set_ylabel("Block minimum")
    axes[0].set_title(f"{system_name}: apparent minimum state over time")
    axes[0].grid(alpha=0.25)

    axes[1].plot(cumulative_minima["end_time"], cumulative_minima["minimum_state"].map(state_to_number), marker="o")
    axes[1].set_yticks(list(state_to_number.values()))
    axes[1].set_yticklabels(list(state_to_number.keys()))
    axes[1].set_ylabel("Cumulative minimum")
    axes[1].grid(alpha=0.25)

    diagnostics = block_results.drop_duplicates("block").sort_values("block")
    axes[2].plot(diagnostics["midpoint_time"], diagnostics["weight_ess_fraction"], marker="o")
    axes[2].set_xlabel("Simulation time")
    axes[2].set_ylabel("Weight ESS / frames")
    axes[2].grid(alpha=0.25)

    fig.tight_layout()
    fig.savefig(output_directory / f"{system_name}_minimum_state_over_time.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
# =====================================================================
# PART 1: RUN ALL FOUR SINGLE TRAJECTORIES
# =====================================================================

single_global_results = {}
single_cumulative_minima = {}

all_system_global_tables = []
all_system_late_tables = []
single_system_summary_records = []

for system_name in SINGLE_SYSTEM_NAMES:
    input_file = SINGLE_INPUT_FILES[system_name]
    system_output_directory = SINGLE_RESULTS_DIRECTORY / system_name
    system_output_directory.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 78)
    print(f"PROCESSING: {system_name}")
    print(f"INPUT:      {input_file}")
    print("=" * 78)

    work = read_plumed_colvar(input_file, stride=ANALYSIS_STRIDE)
    work = assign_states(work)

    print(f"Frames after stride: {len(work):,}")
    print(f"Time range: {work[TIME_COLUMN].iloc[0]} to {work[TIME_COLUMN].iloc[-1]}")

    # ------------------------------------------------------------ global
    global_results = pd.DataFrame(
        analyze_interval(work, start=0, stop=len(work), label="global")
    )
    global_results.insert(0, "system", system_name)
    global_results = global_results.sort_values("delta_free_energy").reset_index(drop=True)

    # ------------------------------------------------------------ block-local
    block_edges = np.linspace(0, len(work), N_BLOCKS + 1, dtype=int)
    block_records = []

    for block_number, (start, stop) in enumerate(zip(block_edges[:-1], block_edges[1:])):
        records = analyze_interval(work, start=start, stop=stop, label=f"block_{block_number:03d}")
        for record in records:
            record["system"] = system_name
            record["block"] = block_number
        block_records.extend(records)

    block_results = pd.DataFrame(block_records)

    block_minima = (
        block_results.loc[block_results.groupby("block")["population"].idxmax()]
        [["system", "block", "start_time", "end_time", "midpoint_time", "state",
          "population", "delta_free_energy", "weight_ess", "weight_ess_fraction"]]
        .rename(columns={"state": "minimum_state", "population": "minimum_state_population"})
        .sort_values("block")
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------ cumulative
    cumulative_results = block_results.sort_values(["state", "block"]).copy()

    for column in ["population", "biased_population"]:
        cumulative_results[column] = (
            cumulative_results.groupby("state")[column]
            .transform(lambda x: x.expanding(min_periods=1).mean())
        )

    cumulative_results = add_free_energies_to_table(cumulative_results, population_column="population")
    cumulative_results = add_free_energies_to_table(cumulative_results, population_column="biased_population")
    cumulative_results = cumulative_results.sort_values(["block", "state"]).reset_index(drop=True)

    cumulative_minima = (
        cumulative_results.loc[cumulative_results.groupby("block")["population"].idxmax()]
        [["system", "block", "start_time", "end_time", "midpoint_time", "state",
          "population", "delta_free_energy"]]
        .rename(columns={"state": "minimum_state", "population": "minimum_state_population"})
        .sort_values("block")
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------ late-block summary
    first_final_block = max(0, N_BLOCKS - N_FINAL_BLOCKS)
    late_blocks = block_results[block_results["block"] >= first_final_block]

    late_summary = late_blocks.groupby("state", as_index=False).agg(
        population=("population", "mean"),
        population_std_across_blocks=("population", "std"),
        biased_population=("biased_population", "mean"),
        biased_population_std_across_blocks=("biased_population", "std"),
        n_contributing_blocks=("block", "nunique"),
    )
    late_summary = add_free_energies_simple(late_summary)
    late_summary.insert(0, "system", system_name)
    late_summary["first_final_block"] = first_final_block
    late_summary["last_final_block"] = N_BLOCKS - 1
    late_summary = late_summary.sort_values("delta_free_energy").reset_index(drop=True)

    # ------------------------------------------------------------ store + save
    single_global_results[system_name] = global_results
    single_cumulative_minima[system_name] = cumulative_minima

    all_system_global_tables.append(global_results)
    all_system_late_tables.append(late_summary)

    global_results.to_csv(system_output_directory / f"{system_name}_global_state_free_energies.csv", index=False)
    block_results.to_csv(system_output_directory / f"{system_name}_block_state_free_energies.csv", index=False)
    cumulative_results.to_csv(system_output_directory / f"{system_name}_cumulative_state_free_energies.csv", index=False)
    late_summary.to_csv(system_output_directory / f"{system_name}_late_state_summary.csv", index=False)
    block_minima.to_csv(system_output_directory / f"{system_name}_block_minimum_over_time.csv", index=False)
    cumulative_minima.to_csv(system_output_directory / f"{system_name}_cumulative_minimum_over_time.csv", index=False)

    global_minimum = global_results.iloc[0]
    late_minimum = late_summary.iloc[0]
    final_cumulative_minimum = cumulative_minima.iloc[-1]

    single_system_summary_records.append({
        "system": system_name,
        "n_analyzed_frames": len(work),
        "start_time": work[TIME_COLUMN].iloc[0],
        "end_time": work[TIME_COLUMN].iloc[-1],
        "global_minimum_state": global_minimum["state"],
        "global_minimum_population": global_minimum["population"],
        "late_minimum_state": late_minimum["state"],
        "late_minimum_population": late_minimum["population"],
        "final_cumulative_minimum_state": final_cumulative_minimum["minimum_state"],
        "final_cumulative_minimum_population": final_cumulative_minimum["minimum_state_population"],
        "global_weight_ess": global_minimum["weight_ess"],
        "global_weight_ess_fraction": global_minimum["weight_ess_fraction"],
    })

    print(f"\nGlobal approximate minimum: {global_minimum['state']} (P = {global_minimum['population']:.4f})")
    print(f"Final-{N_FINAL_BLOCKS}-block minimum: {late_minimum['state']} (P = {late_minimum['population']:.4f})")

    create_single_trajectory_plots(
        system_name=system_name, global_results=global_results, block_results=block_results,
        cumulative_results=cumulative_results, block_minima=block_minima,
        cumulative_minima=cumulative_minima, output_directory=system_output_directory,
    )

    del work
    gc.collect()

In [ ]:
# =====================================================================
# PART 1: COMBINE AND SAVE ACROSS THE FOUR SYSTEMS -- NO AVERAGING
# =====================================================================

all_single_global_results = pd.concat(all_system_global_tables, ignore_index=True)
all_single_late_results = pd.concat(all_system_late_tables, ignore_index=True)
single_system_summary = pd.DataFrame(single_system_summary_records)

all_single_global_results.to_csv(SINGLE_RESULTS_DIRECTORY / "all_single_trajectories_global_results.csv", index=False)
all_single_late_results.to_csv(SINGLE_RESULTS_DIRECTORY / "all_single_trajectories_late_results.csv", index=False)
single_system_summary.to_csv(SINGLE_RESULTS_DIRECTORY / "single_trajectory_minimum_summary.csv", index=False)

print("\n" + "=" * 88)
print("SINGLE-TRAJECTORY MINIMUM SUMMARY")
print("=" * 88)
print(
    single_system_summary[
        ["system", "global_minimum_state", "global_minimum_population",
         "late_minimum_state", "late_minimum_population",
         "final_cumulative_minimum_state", "final_cumulative_minimum_population",
         "global_weight_ess_fraction"]
    ].to_string(index=False)
)
print(f"\nResults saved in:\n{SINGLE_RESULTS_DIRECTORY}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.ravel()

for ax, system_name in zip(axes, SINGLE_SYSTEM_NAMES):
    plot_data = single_global_results[system_name].set_index("state").reindex(ALL_STATES)
    ax.bar(ALL_STATES, plot_data["delta_free_energy"], color=[STATE_COLORS[s] for s in ALL_STATES])
    ax.set_title(system_name)
    ax.set_xlabel("Cis/trans macrostate")
    ax.set_ylabel(r"Approximate global $\Delta F$ (kJ/mol)")
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Global raw-bias-reweighted macrostate free energies\nSingle trajectories; no averaging", y=1.02)
fig.tight_layout()
fig.savefig(SINGLE_RESULTS_DIRECTORY / "single_trajectories_global_free_energy_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## Part 2: 8-replica ensembles (omega, oppc)

Each ensemble is eight independent walkers started from different initial
configurations. Unlike Part 1, populations here are also averaged **across**
replicas, with the standard error of the mean (SEM) across the eight
independent runs reported alongside the block-to-block spread within a
single run. The two ensembles use different raw file layouts (`oppc` was
run with `COLVAR.<n>` output; `omega` was run with `output.txt`), so each
gets its own file-reading function, but both are processed with the exact
same `process_replica_ensemble` pipeline below. Outputs are written to
`<system>_state_analysis/state_analysis_results/`.

In [ ]:
# =====================================================================
# PART 2 SETTINGS
# =====================================================================

def oppc_replica_file(base_directory, replica_number):
    """folder 1 -> COLVAR.0, folder 2 -> COLVAR.1, ..., folder 8 -> COLVAR.7"""
    colvar_number = replica_number - 1
    return base_directory / str(replica_number) / f"COLVAR.{colvar_number}"


def omega_replica_file(base_directory, replica_number):
    """folder 1..8 -> output.txt"""
    return base_directory / str(replica_number) / "output.txt"


REPLICA_ENSEMBLES = {
    "oppc": {
        "base_directory": Path("/Volumes/LaCie/STEPs/ConvergenceTest/walkers"),
        "get_input_file": oppc_replica_file,
        "results_directory": Path("oppc_state_analysis") / "state_analysis_results",
    },
    "omega": {
        "base_directory": Path("/Volumes/LaCie/STEPs/ConvergenceTest/walkers2"),
        "get_input_file": omega_replica_file,
        "results_directory": Path("omega_state_analysis") / "state_analysis_results",
    },
}

REPLICA_NUMBERS = range(1, 9)

In [ ]:
def process_replica_ensemble(ensemble_name, config, start_block=0):
    """
    Run the full 8-replica reweighting + convergence pipeline for one
    ensemble (config holds base_directory / get_input_file / results_directory,
    see REPLICA_ENSEMBLES above). Mirrors the single-trajectory pipeline in
    Part 1, plus an across-replica averaging step.
    """
    base_directory = config["base_directory"]
    get_input_file = config["get_input_file"]
    results_directory = config["results_directory"]
    results_directory.mkdir(parents=True, exist_ok=True)

    input_files = {n: get_input_file(base_directory, n) for n in REPLICA_NUMBERS}

    print("\n" + "#" * 78)
    print(f"ENSEMBLE: {ensemble_name}")
    print("#" * 78)
    print("Input-file mapping:")
    missing_files = []
    for replica_number, input_file in input_files.items():
        status = "FOUND" if input_file.exists() else "MISSING"
        print(f"Replica {replica_number}: {input_file} [{status}]")
        if not input_file.exists():
            missing_files.append(input_file)

    if missing_files:
        raise FileNotFoundError(
            "The following input files were not found (mount the LaCie drive first):\n"
            + "\n".join(str(p) for p in missing_files)
        )

    # ------------------------------------------------------------ per-replica
    all_global_results = []
    all_block_results = []
    all_late_results = []
    processing_summary = []

    for replica_number in REPLICA_NUMBERS:
        replica_name = f"replica_{replica_number}"
        input_file = input_files[replica_number]

        print("\n" + "=" * 72)
        print(f"PROCESSING {ensemble_name}/{replica_name}")
        print(f"Input: {input_file}")
        print("=" * 72)

        work = read_plumed_colvar(input_file, stride=ANALYSIS_STRIDE)
        work = assign_states(work)

        print(f"Frames after stride: {len(work):,}")
        print(f"Time range: {work[TIME_COLUMN].iloc[0]} to {work[TIME_COLUMN].iloc[-1]}")

        replica_global = pd.DataFrame(
            analyze_interval(work, start=0, stop=len(work), label="global")
        )
        replica_global["replica"] = replica_name
        replica_global["replica_number"] = replica_number
        replica_global = replica_global.sort_values("delta_free_energy").reset_index(drop=True)

        block_edges = np.linspace(0, len(work), N_BLOCKS + 1, dtype=int)
        block_records = []
        for block_number, (start, stop) in enumerate(zip(block_edges[:-1], block_edges[1:])):
            records = analyze_interval(work, start=start, stop=stop, label=f"block_{block_number:03d}")
            for record in records:
                record["block"] = block_number
                record["replica"] = replica_name
                record["replica_number"] = replica_number
            block_records.extend(records)

        replica_blocks = pd.DataFrame(block_records)

        first_final_block = max(0, N_BLOCKS - N_FINAL_BLOCKS)
        final_block_rows = replica_blocks[replica_blocks["block"] >= first_final_block].copy()

        replica_late = final_block_rows.groupby("state", as_index=False).agg(
            population=("population", "mean"),
            population_std_across_blocks=("population", "std"),
            biased_population=("biased_population", "mean"),
            biased_population_std_across_blocks=("biased_population", "std"),
            n_contributing_blocks=("block", "nunique"),
        )
        replica_late = add_free_energies_simple(replica_late)
        replica_late.insert(0, "replica", replica_name)
        replica_late.insert(1, "replica_number", replica_number)
        replica_late["first_final_block"] = first_final_block
        replica_late["last_final_block"] = N_BLOCKS - 1
        replica_late = replica_late.sort_values("delta_free_energy").reset_index(drop=True)

        replica_global.to_csv(results_directory / f"{replica_name}_global_state_free_energies.csv", index=False)
        replica_blocks.to_csv(results_directory / f"{replica_name}_block_state_free_energies.csv", index=False)
        replica_late.to_csv(results_directory / f"{replica_name}_late_state_summary.csv", index=False)

        all_global_results.append(replica_global)
        all_block_results.append(replica_blocks)
        all_late_results.append(replica_late)

        global_minimum = replica_global.iloc[0]
        late_minimum = replica_late.iloc[0]

        processing_summary.append({
            "replica": replica_name,
            "replica_number": replica_number,
            "n_frames": len(work),
            "start_time": work[TIME_COLUMN].iloc[0],
            "end_time": work[TIME_COLUMN].iloc[-1],
            "global_minimum_state": global_minimum["state"],
            "global_minimum_population": global_minimum["population"],
            "late_minimum_state": late_minimum["state"],
            "late_minimum_population": late_minimum["population"],
            "global_weight_ess": global_minimum["weight_ess"],
            "global_weight_ess_fraction": global_minimum["weight_ess_fraction"],
        })

        print(f"Global approximate minimum: {global_minimum['state']} (P = {global_minimum['population']:.4f})")
        print(f"Late-block minimum:          {late_minimum['state']} (P = {late_minimum['population']:.4f})")

        del work
        gc.collect()

    all_global_results = pd.concat(all_global_results, ignore_index=True)
    all_block_results = pd.concat(all_block_results, ignore_index=True)
    all_late_results = pd.concat(all_late_results, ignore_index=True)
    processing_summary = pd.DataFrame(processing_summary)

    all_global_results.to_csv(results_directory / "all_replicas_global_results.csv", index=False)
    all_block_results.to_csv(results_directory / "all_replicas_block_results.csv", index=False)
    all_late_results.to_csv(results_directory / "all_replicas_late_results.csv", index=False)
    processing_summary.to_csv(results_directory / "all_replicas_minimum_summary.csv", index=False)

    # ------------------------------------------------------------ across-replica averages
    def average_across_replicas(table, source_population_column):
        averaged = table.groupby("state", as_index=False).agg(
            population=(source_population_column, "mean"),
            population_std_across_replicas=(source_population_column, "std"),
            population_sem_across_replicas=(
                source_population_column,
                lambda x: x.std(ddof=1) / np.sqrt(x.count()) if x.count() > 1 else np.nan,
            ),
            biased_population=("biased_population", "mean"),
            n_replicas=("replica", "nunique"),
        )
        averaged = add_free_energies_simple(averaged)
        return averaged.sort_values("delta_free_energy").reset_index(drop=True)

    global_replica_average = average_across_replicas(all_global_results, "population")
    late_replica_average = average_across_replicas(all_late_results, "population")

    global_replica_average.to_csv(results_directory / "all_replicas_global_population_average.csv", index=False)
    late_replica_average.to_csv(results_directory / "all_replicas_late_population_average.csv", index=False)

    print("\n" + "=" * 72)
    print(f"{ensemble_name}: FINAL-BLOCK POPULATIONS AVERAGED ACROSS REPLICAS")
    print("=" * 72)
    print(
        late_replica_average[
            ["state", "population", "population_std_across_replicas",
             "population_sem_across_replicas", "delta_free_energy", "biased_population", "n_replicas"]
        ].to_string(index=False)
    )

    # ------------------------------------------------------------ population convergence over time
    time_results = (
        all_block_results[all_block_results["block"] >= start_block]
        .copy()
        .sort_values(["replica_number", "state", "block"])
        .reset_index(drop=True)
    )

    for column, cumulative_column in [
        ("population", "cumulative_population"),
        ("biased_population", "cumulative_biased_population"),
    ]:
        time_results[cumulative_column] = (
            time_results.groupby(["replica_number", "state"], sort=False)[column]
            .transform(lambda x: x.expanding(min_periods=1).mean())
        )
        time_results[cumulative_column] = (
            time_results[cumulative_column]
            / time_results.groupby(["replica_number", "block"])[cumulative_column].transform("sum")
        )

    def sem(x):
        return x.std(ddof=1) / np.sqrt(x.count()) if x.count() > 1 else np.nan

    ensemble_cumulative = time_results.groupby(["block", "state"], as_index=False).agg(
        start_time=("start_time", "mean"),
        end_time=("end_time", "mean"),
        midpoint_time=("midpoint_time", "mean"),
        population=("cumulative_population", "mean"),
        population_sd=("cumulative_population", "std"),
        population_sem=("cumulative_population", sem),
        biased_population=("cumulative_biased_population", "mean"),
        biased_population_sd=("cumulative_biased_population", "std"),
        biased_population_sem=("cumulative_biased_population", sem),
        n_replicas=("replica_number", "nunique"),
    )

    for column in ["population", "biased_population"]:
        ensemble_cumulative[column] = (
            ensemble_cumulative[column] / ensemble_cumulative.groupby("block")[column].transform("sum")
        )

    for population_column, fe_column, delta_column in [
        ("population", "free_energy", "delta_free_energy"),
        ("biased_population", "biased_free_energy", "biased_delta_free_energy"),
    ]:
        ensemble_cumulative[fe_column] = np.where(
            ensemble_cumulative[population_column] > 0,
            -K_B * TEMPERATURE * np.log(ensemble_cumulative[population_column]),
            np.inf,
        )
        ensemble_cumulative[delta_column] = (
            ensemble_cumulative[fe_column] - ensemble_cumulative.groupby("block")[fe_column].transform("min")
        )

    minimum_state_over_time = (
        ensemble_cumulative.loc[ensemble_cumulative.groupby("block")["population"].idxmax()]
        [["block", "end_time", "state", "population", "delta_free_energy"]]
        .rename(columns={"state": "minimum_state", "population": "minimum_state_population"})
        .reset_index(drop=True)
    )

    final_block_number = ensemble_cumulative["block"].max()
    final_cumulative = (
        ensemble_cumulative[ensemble_cumulative["block"] == final_block_number]
        .sort_values("delta_free_energy")
        .reset_index(drop=True)
    )

    time_results.to_csv(results_directory / f"per_replica_cumulative_populations_start{start_block}.csv", index=False)
    ensemble_cumulative.to_csv(results_directory / f"ensemble_cumulative_populations_start{start_block}.csv", index=False)
    final_cumulative.to_csv(results_directory / f"final_cumulative_populations_start{start_block}.csv", index=False)
    minimum_state_over_time.to_csv(results_directory / f"minimum_state_over_time_start{start_block}.csv", index=False)

    print("\nApparent minimum state over time:")
    print(minimum_state_over_time.to_string(index=False))
    print(f"\nResults saved in:\n{results_directory}")

    return {
        "processing_summary": processing_summary,
        "all_late_results": all_late_results,
        "late_replica_average": late_replica_average,
        "ensemble_cumulative": ensemble_cumulative,
        "time_results": time_results,
        "results_directory": results_directory,
    }

In [ ]:
def plot_replica_ensemble(ensemble_name, results):
    """Late-population summary and convergence-over-time plots for one ensemble."""
    results_directory = results["results_directory"]
    late_replica_average = results["late_replica_average"]
    all_late_results = results["all_late_results"]
    ensemble_cumulative = results["ensemble_cumulative"]
    time_results = results["time_results"]

    late_plot = late_replica_average.set_index("state").reindex(ALL_STATES)
    colors = [STATE_COLORS[s] for s in ALL_STATES]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(
        ALL_STATES, late_plot["population"], yerr=late_plot["population_sem_across_replicas"],
        capsize=4, color=colors,
    )
    axes[0].set_xlabel("Cis/trans macrostate")
    axes[0].set_ylabel("Mean late-block population")
    axes[0].set_title(f"{ensemble_name}: mean across 8 replicas\nError bars: SEM across replicas")
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].bar(ALL_STATES, late_plot["delta_free_energy"], color=colors)
    axes[1].set_xlabel("Cis/trans macrostate")
    axes[1].set_ylabel(r"$\Delta F$ (kJ/mol)")
    axes[1].set_title(f"{ensemble_name}: free energy from mean replica population")
    axes[1].grid(axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(results_directory / "all_replicas_late_population_and_free_energy.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    replica_population_pivot = (
        all_late_results.pivot(index="replica_number", columns="state", values="population")
        .reindex(columns=ALL_STATES)
    )

    fig, ax = plt.subplots(figsize=(11, 6))
    for state in ALL_STATES:
        ax.plot(replica_population_pivot.index, replica_population_pivot[state], marker="o", linewidth=1.5, label=state)
    ax.set_xlabel("Replica number")
    ax.set_ylabel("Mean population in final blocks")
    ax.set_xticks(list(REPLICA_NUMBERS))
    ax.set_ylim(bottom=0)
    ax.set_title(f"{ensemble_name}: late-block populations for each replica")
    ax.grid(alpha=0.25)
    ax.legend(title="State", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(results_directory / "late_populations_by_replica.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(3, 1, figsize=(12, 15), sharex=True)

    for state in ALL_STATES:
        subset = ensemble_cumulative[ensemble_cumulative["state"] == state].sort_values("block")
        x = subset["end_time"].to_numpy()
        y = subset["population"].to_numpy()
        sem = subset["population_sem"].to_numpy()

        axes[0].plot(x, y, marker="o", markersize=3, linewidth=1.8, color=STATE_COLORS[state], label=state)
        axes[0].fill_between(x, np.clip(y - sem, 0, 1), np.clip(y + sem, 0, 1), color=STATE_COLORS[state], alpha=0.12)

    axes[0].set_ylabel("Cumulative population")
    axes[0].set_ylim(0, 1)
    axes[0].set_title(f"{ensemble_name}: cumulative convergence of block-averaged populations\nShading: SEM across replicas")
    axes[0].grid(alpha=0.25)

    for state in ALL_STATES:
        subset = ensemble_cumulative[ensemble_cumulative["state"] == state].sort_values("block")
        axes[1].plot(subset["end_time"], subset["delta_free_energy"], marker="o", markersize=3, linewidth=1.8, color=STATE_COLORS[state], label=state)

    axes[1].set_ylabel(r"Cumulative $\Delta F$ (kJ mol$^{-1}$)")
    axes[1].set_title(f"{ensemble_name}: relative macrostate free energies from cumulative populations")
    axes[1].grid(alpha=0.25)

    diagnostics = time_results.groupby("block", as_index=False)["weight_ess_fraction"].mean()
    axes[2].plot(
        time_results.groupby("block")["midpoint_time"].mean(), diagnostics["weight_ess_fraction"], marker="o",
    )
    axes[2].set_xlabel("Simulation time")
    axes[2].set_ylabel("Mean weight ESS / frames\n(averaged across replicas)")
    axes[2].grid(alpha=0.25)

    axes[0].legend(title="State", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(results_directory / "population_convergence_start0.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(4, 2, figsize=(14, 16), sharex=True, sharey=True)
    axes = axes.ravel()

    for ax, state in zip(axes, ALL_STATES):
        state_data = time_results[time_results["state"] == state]

        for replica_number in sorted(state_data["replica_number"].unique()):
            subset = state_data[state_data["replica_number"] == replica_number].sort_values("block")
            ax.plot(subset["end_time"], subset["cumulative_population"], linewidth=1.0, alpha=0.55, label=f"Replica {replica_number}")

        mean_subset = ensemble_cumulative[ensemble_cumulative["state"] == state].sort_values("block")
        ax.plot(mean_subset["end_time"], mean_subset["population"], color="black", linewidth=2.5, label="Replica mean")

        ax.set_title(state)
        ax.set_ylim(0, 0.55)
        ax.grid(alpha=0.25)

    for ax in axes[-2:]:
        ax.set_xlabel("Simulation time")
    for ax in axes[::2]:
        ax.set_ylabel("Cumulative population")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.01))
    fig.suptitle(f"{ensemble_name}: cumulative population convergence in each independent replica", y=1.035)
    fig.tight_layout()
    fig.savefig(results_directory / "per_replica_population_convergence_start0.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
# =====================================================================
# PART 2: RUN BOTH 8-REPLICA ENSEMBLES
# =====================================================================

replica_ensemble_results = {}

for ensemble_name, config in REPLICA_ENSEMBLES.items():
    replica_ensemble_results[ensemble_name] = process_replica_ensemble(ensemble_name, config, start_block=0)
    plot_replica_ensemble(ensemble_name, replica_ensemble_results[ensemble_name])

## Next step

Run [convergence_plotting.ipynb](convergence_plotting.ipynb) or
[single_trajectory_state_analysis.ipynb](single_trajectory_state_analysis.ipynb)
to load the CSVs written above.